In [1]:
!pip install groq -q

import os
import json
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

print("Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.0 MB/s eta 0:00:00
Ready!


In [2]:
os.makedirs("agent", exist_ok=True)

treatment_data = {
    "Apple Scab": {
        "organic": [
            "Apply neem oil spray every 7-10 days",
            "Use sulfur-based fungicide before infection",
            "Remove and destroy fallen infected leaves"
        ],
        "chemical": [
            "Apply Captan fungicide at bud break",
            "Use Myclobutanil (Rally) during growing season",
            "Spray Mancozeb every 7 days in wet conditions"
        ],
        "prevention": "Plant resistant apple varieties, ensure good air circulation"
    },
    "Late blight": {
        "organic": [
            "Apply copper-based fungicide immediately",
            "Remove and destroy all infected plant parts",
            "Avoid overhead irrigation — use drip instead"
        ],
        "chemical": [
            "Apply Chlorothalonil (Daconil) at first sign",
            "Use Mefenoxam (Ridomil) for soil drenching",
            "Rotate with Cymoxanil for resistance management"
        ],
        "prevention": "Use certified disease-free seeds, avoid wet foliage"
    },
    "Early blight": {
        "organic": [
            "Apply neem oil or copper spray weekly",
            "Mulch around base to prevent soil splash",
            "Remove lower infected leaves immediately"
        ],
        "chemical": [
            "Apply Chlorothalonil every 7-10 days",
            "Use Azoxystrobin (Quadris) for systemic protection",
            "Alternate fungicides to prevent resistance"
        ],
        "prevention": "Crop rotation every 2-3 years, stake plants for airflow"
    },
    "Powdery mildew": {
        "organic": [
            "Spray diluted baking soda solution (1 tbsp per litre)",
            "Apply potassium bicarbonate spray",
            "Use neem oil weekly as preventive"
        ],
        "chemical": [
            "Apply Myclobutanil (Eagle) at first sign",
            "Use Trifloxystrobin (Flint) for long protection",
            "Spray Tebuconazole every 14 days"
        ],
        "prevention": "Avoid over-fertilizing with nitrogen, improve air circulation"
    },
    "Northern Leaf Blight": {
        "organic": [
            "Apply copper hydroxide spray at early infection",
            "Remove heavily infected leaves",
            "Ensure proper plant spacing for airflow"
        ],
        "chemical": [
            "Apply Propiconazole (Tilt) at tasseling stage",
            "Use Azoxystrobin + Propiconazole (Quilt) mixture",
            "Spray Pyraclostrobin (Headline) for best control"
        ],
        "prevention": "Plant resistant corn hybrids, rotate crops annually"
    },
    "Tomato Yellow Leaf Curl Virus": {
        "organic": [
            "Use reflective silver mulch to repel whiteflies",
            "Install yellow sticky traps around plants",
            "Spray neem oil to reduce whitefly population"
        ],
        "chemical": [
            "Apply Imidacloprid (Confidor) to control whiteflies",
            "Use Thiamethoxam (Actara) as systemic insecticide",
            "Spray Spiromesifen (Oberon) for whitefly larvae"
        ],
        "prevention": "Use virus-resistant tomato varieties, remove infected plants immediately"
    },
    "healthy": {
        "organic": [
            "Continue regular compost application",
            "Use neem oil as preventive spray monthly",
            "Maintain proper watering schedule"
        ],
        "chemical": [],
        "prevention": "Regular monitoring, balanced fertilization, proper spacing"
    }
}

with open("agent/treatment_data.json", "w") as f:
    json.dump(treatment_data, f, indent=2)

print(f"treatment_data.json saved with {len(treatment_data)} diseases!")

treatment_data.json saved with 7 diseases!


In [3]:
def treatment_advice(disease_name, farming_type="both"):
    """
    Tool 2: Given a disease name, returns organic + chemical treatments.
    farming_type: 'organic', 'chemical', or 'both'
    """
    with open("agent/treatment_data.json", "r") as f:
        data = json.load(f)

    # Find matching disease
    matched_key = None
    if disease_name in data:
        matched_key = disease_name
    else:
        for key in data:
            if key.lower() in disease_name.lower() or disease_name.lower() in key.lower():
                matched_key = key
                break

    if not matched_key:
        return {
            "disease":    disease_name,
            "organic":    ["Consult local agricultural extension officer"],
            "chemical":   ["Consult local agricultural extension officer"],
            "prevention": "No specific data available",
            "found":      False
        }

    info = data[matched_key]

    # Filter based on farming type
    result = {
        "disease":    matched_key,
        "prevention": info["prevention"],
        "found":      True
    }

    if farming_type in ("organic", "both"):
        result["organic"] = info["organic"]
    if farming_type in ("chemical", "both"):
        result["chemical"] = info["chemical"]

    return result

# Test it
result = treatment_advice("Late blight")
print(json.dumps(result, indent=2))

{
  "disease": "Late blight",
  "prevention": "Use certified disease-free seeds, avoid wet foliage",
  "found": true,
  "organic": [
    "Apply copper-based fungicide immediately",
    "Remove and destroy all infected plant parts",
    "Avoid overhead irrigation \u2014 use drip instead"
  ],
  "chemical": [
    "Apply Chlorothalonil (Daconil) at first sign",
    "Use Mefenoxam (Ridomil) for soil drenching",
    "Rotate with Cymoxanil for resistance management"
  ]
}


In [4]:
# Recreate disease_data.json since this is a new session
disease_data = {
    "Apple Scab": {
        "cause": "Fungal infection caused by Venturia inaequalis",
        "symptoms": "Dark olive-green spots on leaves, eventually turning brown and scabby",
        "severity": "Medium"
    },
    "Late blight": {
        "cause": "Water mold Phytophthora infestans",
        "symptoms": "Dark water-soaked lesions on leaves and stems, white mold on underside",
        "severity": "High"
    },
    "Early blight": {
        "cause": "Fungal infection by Alternaria solani",
        "symptoms": "Dark brown spots with concentric rings, yellow halo around spots",
        "severity": "Medium"
    },
    "Powdery mildew": {
        "cause": "Various fungal species depending on host plant",
        "symptoms": "White powdery coating on leaf surface, distorted young leaves",
        "severity": "Low"
    },
    "Northern Leaf Blight": {
        "cause": "Fungal infection by Exserohilum turcicum",
        "symptoms": "Long greyish-green lesions on corn leaves, cigar-shaped spots",
        "severity": "Medium"
    },
    "Tomato Yellow Leaf Curl Virus": {
        "cause": "Virus transmitted by whitefly Bemisia tabaci",
        "symptoms": "Yellowing and upward curling of leaves, stunted plant growth",
        "severity": "High"
    },
    "healthy": {
        "cause": "No disease detected",
        "symptoms": "Plant appears healthy with no visible symptoms",
        "severity": "None"
    }
}

with open("agent/disease_data.json", "w") as f:
    json.dump(disease_data, f, indent=2)

def disease_info(disease_name):
    with open("agent/disease_data.json", "r") as f:
        data = json.load(f)
    if disease_name in data:
        info = data[disease_name]
        return {"disease": disease_name, "cause": info["cause"],
                "symptoms": info["symptoms"], "severity": info["severity"], "found": True}
    for key in data:
        if key.lower() in disease_name.lower() or disease_name.lower() in key.lower():
            info = data[key]
            return {"disease": key, "cause": info["cause"],
                    "symptoms": info["symptoms"], "severity": info["severity"], "found": True}
    return {"disease": disease_name, "cause": "Unknown", "symptoms": "Unknown",
            "severity": "Unknown", "found": False}

print("disease_info tool ready!")

disease_info tool ready!


In [5]:
def run_agent(disease_name, farming_type="both"):
    """
    Agent with 2 tools:
    Tool 1 — disease_info()     → cause, symptoms, severity
    Tool 2 — treatment_advice() → organic + chemical treatments
    Agent calls BOTH then gives complete structured response.
    """

    # Call both tools
    info      = disease_info(disease_name)
    treatment = treatment_advice(disease_name, farming_type)

    # Build organic treatment string
    organic_str  = "\n".join([f"  - {t}" for t in treatment.get("organic", [])])
    chemical_str = "\n".join([f"  - {t}" for t in treatment.get("chemical", [])])

    system_prompt = """You are an expert agricultural advisor helping farmers
diagnose and treat crop diseases. Be concise, practical, and compassionate.
The farmer may be losing their crop — give clear, actionable advice."""

    user_message = f"""
A farmer's crop has been diagnosed with: {info['disease']}

DISEASE INFORMATION:
- Cause: {info['cause']}
- Symptoms: {info['symptoms']}
- Severity: {info['severity']}

TREATMENT OPTIONS:
Organic treatments:
{organic_str}

Chemical treatments:
{chemical_str}

Prevention for future:
{treatment['prevention']}

Give the farmer a complete, structured diagnosis report with:
1. What this disease is and how serious it is
2. Organic treatment steps (if they prefer organic farming)
3. Chemical treatment steps (if they need faster results)
4. Prevention advice for next season
Keep it practical and under 200 words.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        max_tokens=400
    )

    return response.choices[0].message.content

print("Agent with 2 tools ready!")

Agent with 2 tools ready!


In [6]:
test_diseases = ["Late blight", "Apple Scab", "healthy"]

for disease in test_diseases:
    print(f"\n{'='*60}")
    print(f"🌿 Disease: {disease}")
    print('='*60)
    print(run_agent(disease))


🌿 Disease: Late blight
**Late Blight Diagnosis Report**

1. **Disease Overview**: Late blight is a serious disease caused by the water mold Phytophthora infestans, characterized by dark water-soaked lesions and white mold. The severity of this outbreak is high, requiring immediate attention.

2. **Organic Treatment**: To manage the disease organically, apply a copper-based fungicide immediately, remove and destroy infected plant parts, and switch to drip irrigation to avoid wet foliage.

3. **Chemical Treatment**: For faster results, apply Chlorothalonil (Daconil) at the first sign of infection, use Mefenoxam (Ridomil) for soil drenching, and rotate with Cymoxanil to manage resistance.

4. **Prevention Advice**: To prevent future outbreaks, use certified disease-free seeds and avoid overhead irrigation to minimize wet foliage. Implementing these prevention strategies will help reduce the risk of late blight in next season's crop.

🌿 Disease: Apple Scab
**Diagnosis Report: Apple Scab**

In [7]:
print("=== Organic farmer asking about Powdery Mildew ===\n")
print(run_agent("Powdery mildew", farming_type="organic"))

=== Organic farmer asking about Powdery Mildew ===

**Diagnosis Report: Powdery Mildew**

1. **Disease Overview**: Powdery mildew is a fungal disease causing a white powdery coating on leaf surfaces and distorted young leaves. With a low severity level, prompt action can prevent further damage.
2. **Organic Treatment**: To manage the disease, spray a diluted baking soda solution (1 tbsp per litre) or apply potassium bicarbonate spray. For prevention, use neem oil weekly.
3. **Chemical Treatment**: Although not specified, conventional fungicides can be used for faster results. However, organic methods are recommended to avoid chemical residues.
4. **Prevention Advice**: For next season, avoid over-fertilizing with nitrogen, as this can promote disease growth. Improve air circulation around plants to reduce moisture and prevent fungal development. By following these steps, you can effectively manage powdery mildew and prevent future outbreaks.
